# A-Polymer Update Strategy Matrix

This notebook runs a concrete experiment matrix for three constant-food A-polymer networks and four SSA/blended update configurations. The output layout is `outputs/<timestamp>/experiment_matrix/`, containing `test_config.csv/xlsx`, `test_result.csv/xlsx`, trajectories, and per-cell cProfile reports with the top 40 cumulative-time entries.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

COMPARE_DIR = Path.cwd()
if COMPARE_DIR.name != 'compare':
    COMPARE_DIR = Path(r'C:/Users/33973/Documents/New project/examples/compare')
sys.path.insert(0, str(COMPARE_DIR))

from experiment_matrix import (
    run_experiment_matrix,
    matrix_run_dir,
    write_config_files,
    load_config_dataframe,
)
from a_polymer_update_matrix import (
    DEFAULT_A_POLYMER_NETWORKS,
    create_a_polymer_update_config,
)


## Configuration Info

Edit this block to control the three networks, wall-clock budget, worker count, and blended parameters.

In [ ]:
NETWORKS = list(DEFAULT_A_POLYMER_NETWORKS)
WALL_SECONDS = 0.2
MAX_STEPS = 1000
SEED = 123
T_END = None
WORKERS = 1
PROFILE_LIMIT = 40
OUTPUT_ROOT = COMPARE_DIR / 'outputs'
TIMESTAMP = None  # None creates a timestamped output directory

BLENDED_I1 = 100.0
BLENDED_I2 = 150.0
BLENDED_DT_CLE = 0.00033981
BLENDED_DT_MACRO = 0.00033981

config_info = pd.DataFrame(
    [
        ('networks', NETWORKS),
        ('wall_seconds', WALL_SECONDS),
        ('max_steps', MAX_STEPS),
        ('seed', SEED),
        ('t_end', T_END),
        ('workers', WORKERS),
        ('profile_limit', PROFILE_LIMIT),
        ('output_root', str(OUTPUT_ROOT)),
        ('timestamp', TIMESTAMP),
        ('blended_i1', BLENDED_I1),
        ('blended_i2', BLENDED_I2),
        ('blended_dt_cle', BLENDED_DT_CLE),
        ('blended_dt_macro', BLENDED_DT_MACRO),
    ],
    columns=['parameter', 'value'],
)
display(config_info)


## Matrix

Rows are method configurations. Columns are network instances. Each cell is a JSON run specification consumed by `run_experiment_matrix`.

In [ ]:
config = create_a_polymer_update_config(
    networks=NETWORKS,
    wall_seconds=WALL_SECONDS,
    max_steps=MAX_STEPS,
    seed=SEED,
    t_end=T_END,
    blended_i1=BLENDED_I1,
    blended_i2=BLENDED_I2,
    blended_dt_cle=BLENDED_DT_CLE,
    blended_dt_macro=BLENDED_DT_MACRO,
)

preview_dir = matrix_run_dir(OUTPUT_ROOT, 'a_polymer_config_preview')
preview_paths = write_config_files(config, preview_dir)
display(config)
preview_paths


## Run And Results

This block executes the matrix. Every enabled cell writes a trajectory plus `outputs/<timestamp>/experiment_matrix/profiles/<config>__<network>__<method>.prof` and the corresponding `_top40.txt` report.

In [ ]:
run_info = run_experiment_matrix(
    config,
    output_root=OUTPUT_ROOT,
    timestamp=TIMESTAMP,
    workers=WORKERS,
    profile=True,
    profile_limit=PROFILE_LIMIT,
)

result = run_info['result']
long_result = run_info['long_result']
display(result)

summary_columns = [
    'status',
    'config_id',
    'network',
    'method',
    'simulation_final_time',
    'n_events',
    'wall_runtime_seconds',
    'trajectory_path',
    'profile_report_path',
    'error',
]
display(long_result[[column for column in summary_columns if column in long_result.columns]])
print(f"run_dir: {run_info['run_dir']}")


## Load Existing Config

To run an edited config file later, load `test_config.csv` and pass it to `run_experiment_matrix`.

In [ ]:
# edited_config = load_config_dataframe(preview_paths['csv'])
# run_experiment_matrix(
#     edited_config,
#     output_root=OUTPUT_ROOT,
#     workers=4,
#     profile=True,
#     profile_limit=40,
# )
